In [1]:
import numpy as np
from datasets import load_dataset

news = load_dataset('argilla/news-summary', split = 'test')
df = news.to_pandas().sample(5000, random_state = 42)[['text', 'prediction']]
df['text'] = 'summarize: ' + df['text']
df['prediction'] = df['prediction'].map(lambda x: x[0]['text'])
train, valid, test = np.split(df.sample(frac = 1, random_state = 42), [int(0.6 * len(df)), int(0.8 * len(df))])

print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Putin says had useful interaction with Trump at Vi
3000
1000
1000


c:\Users\use\Documents\KDT17\cv-deep-learning\.venv\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [8]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    source = tokenizer(text = data.text.tolist(), padding = 'max_length', max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    target = tokenizer(text = data.text.tolist(), padding = 'max_length', max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    source_ids = source['input_ids'].squeeze().to(device)
    source_mask = source['attention_mask'].squeeze().to(device)
    target_ids = target['input_ids'].squeeze().to(device)
    target_mask = target['attention_mask'].squeeze().to(device)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset,
                            sampler = data_sampler,
                            batch_size = batch_size)
    return dataloader

In [3]:
epochs = 5
batch_size = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path = "t5-small")

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[tensor([[21603,    10,  8747,  ..., 18407,  4026,     1],
        [21603,    10,  6045,  ...,    37, 16870,     1],
        [21603,    10,  4753,  ...,    37, 12371,     1],
        ...,
        [21603,    10,    71,  ...,    29, 10524,     1],
        [21603,    10,     3,  ...,     5,  4623,     1],
        [21603,    10,     3,  ...,     6,    62,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[21603,    10,  8747,  ..., 18407,  4026,     1],
        [21603,    10,  6045,  ...,    37, 16870,     1],
        [21603,    10,  4753,  ...,    37, 12371,     1],
        ...,
        [21603,    10,    71,  ...,    29, 10524,     1],
        [21603,    10,     3,  ...,     5,  4623,     1],
        [21603,    10,     3,  ...,     6,    62,     1]], device='cuda:0'), ten

In [4]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path = 't5-small').to(device)

optimizer = optim.AdamW(model.parameters(), lr = 1e-5, eps = 1e-8)

In [5]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis = 1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask = source_mask,
                        decoder_input_ids = decoder_input_ids, labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)

    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(input_ids = source_ids, attention_mask = source_mask,
                            decoder_input_ids = decoder_input_ids, labels = labels)
            loss = outputs.loss
            val_loss += loss.item()

        val_loss = val_loss / len(dataloader)

    return val_loss
        

In [6]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)

    print(epoch + 1, train_loss, val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "./models/T5ForConditionalGeneration.pt")
        print("Saved!!")

1 0.9399413666192521 0.2125696954317391
Saved!!
2 0.33071005598027653 0.0798425858374685
Saved!!
3 0.1742097204352947 0.047083182260394096
Saved!!
4 0.11463163634564014 0.027643755660392344
Saved!!
5 0.08221752858383859 0.018023649143287912
Saved!!


In [10]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        generated_ids = model.generate(input_ids = source_ids, attention_mask = source_mask, max_length = 128, num_beams = 3, repetition_penalty = 2.5, length_penalty = 1.0, early_stopping = True)
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(generated, skip_special_tokens = True, clean_up_tokenization_spaces = True)
            actual = tokenizer.decode(target, skip_special_tokens = True, clean_up_tokenization_spaces = True)
            print(pred)
            print(actual)
        break

WASHINGTON (Reuters) - Democratic presidential nominee Hillary Clinton leads Republican Donald Trump by 4 percentage points in a four-war race for the Nov. 8 election, according to a Washington Post-ABC News opinion poll of likely voters released on Friday. Clinton had 47 percent support compared with Trump’s 43 percent in the poll conducted from Monday to Thursday, the Post said. It said Clinton’s lead was up from 3 points in the previous day’s poll but still “within the range of sampling error.”
summarize: WASHINGTON (Reuters) - Democratic presidential nominee Hillary Clinton leads Republican Donald Trump by 4 percentage points in a four-war race for the Nov. 8 election, according to a Washington Post-ABC News opinion poll of likely voters released on Friday. Clinton had 47 percent support compared with Trump’s 43 percent in the poll conducted from Monday to Thursday, the Post said. It said Clinton’s lead was up from 3 points in the previous day’s poll but still “within the range of 